In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mlss-sp-25-classification/sample_submission.csv
/kaggle/input/mlss-sp-25-classification/train.csv
/kaggle/input/mlss-sp-25-classification/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/mlss-sp-25-classification/train.csv")
test = pd.read_csv("/kaggle/input/mlss-sp-25-classification/test.csv")
sample = pd.read_csv("/kaggle/input/mlss-sp-25-classification/sample_submission.csv")
ID = test["id"]
test = test.drop("id", axis = 1)
train = train.drop("id", axis = 1)

In [3]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from sklearn.preprocessing import SplineTransformer

In [4]:
train.head()

,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,...,soil_type32,soil_type33,soil_type34,soil_type35,soil_type36,soil_type37,soil_type38,soil_type39,soil_type40,cover_type
0,2840,20,6,42,6,566,216,228,149,566,...,0,0,0,0,0,0,0,0,0,2
1,2690,95,11,0,0,1605,238,223,114,2346,...,0,0,0,0,0,0,0,0,0,2
2,2759,22,17,0,0,752,207,200,126,2082,...,0,0,0,0,0,0,0,0,0,1
3,3140,51,27,400,219,1981,222,172,68,2754,...,0,1,0,0,0,0,0,0,0,1
4,3170,29,6,30,1,1288,218,226,144,1205,...,0,0,0,0,0,0,0,0,0,2


In [5]:
test.head()

,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,...,soil_type31,soil_type32,soil_type33,soil_type34,soil_type35,soil_type36,soil_type37,soil_type38,soil_type39,soil_type40
0,3089,355,5,630,88,2144,211,230,157,2547,...,0,1,0,0,0,0,0,0,0,0
1,3165,56,10,510,-29,797,227,218,124,1341,...,0,0,0,0,0,0,0,0,0,0
2,3142,23,8,633,125,1674,216,223,144,2608,...,1,0,0,0,0,0,0,0,0,0
3,2782,330,23,85,7,1368,158,204,182,1565,...,0,0,0,0,0,0,0,0,0,0
4,2775,36,14,150,24,1316,218,208,123,2255,...,0,1,0,0,0,0,0,0,0,0


In [6]:
train.isnull().sum().sum()

0

In [7]:
test.isnull().sum().sum()

0

In [8]:
train.columns

Index(['elevation', 'aspect', 'slope', 'horizontal_distance_to_hydrology',
       'vertical_distance_to_hydrology', 'horizontal_distance_to_roadways',
       'hillshade_9am', 'hillshade_noon', 'hillshade_3pm',
       'horizontal_distance_to_fire_points', 'wilderness_area1',
       'wilderness_area2', 'wilderness_area3', 'wilderness_area4',
       'soil_type1', 'soil_type2', 'soil_type3', 'soil_type4', 'soil_type5',
       'soil_type6', 'soil_type7', 'soil_type8', 'soil_type9', 'soil_type10',
       'soil_type11', 'soil_type12', 'soil_type13', 'soil_type14',
       'soil_type15', 'soil_type16', 'soil_type17', 'soil_type18',
       'soil_type19', 'soil_type20', 'soil_type21', 'soil_type22',
       'soil_type23', 'soil_type24', 'soil_type25', 'soil_type26',
       'soil_type27', 'soil_type28', 'soil_type29', 'soil_type30',
       'soil_type31', 'soil_type32', 'soil_type33', 'soil_type34',
       'soil_type35', 'soil_type36', 'soil_type37', 'soil_type38',
       'soil_type39', 'soil_type40

The model is taking too much time to train, so as per your instructions I am only using continuous  variables and also performing random sampling on the dataset to select only 10000 data points.

In [9]:
train = train.sample(n=10000, random_state=42)

In [10]:
X_train = train[['elevation', 'aspect', 'slope', 'horizontal_distance_to_hydrology', 'vertical_distance_to_hydrology', 'horizontal_distance_to_roadways', 'hillshade_9am', 'hillshade_noon', 'hillshade_3pm', 'horizontal_distance_to_fire_points']]
y_train = train["cover_type"]
X_test = test[['elevation', 'aspect', 'slope', 'horizontal_distance_to_hydrology', 'vertical_distance_to_hydrology', 'horizontal_distance_to_roadways', 'hillshade_9am', 'hillshade_noon', 'hillshade_3pm', 'horizontal_distance_to_fire_points']]

In [11]:
assert set(X_train.columns) == set(X_test.columns)

In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
f1_macro = make_scorer(f1_score, average="macro")

In [13]:
penalty_type = ['l1', 'l2']
C_values = np.logspace(-3, 5, 9)

logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(solver='liblinear'))
])

param_grid = {
    'logreg__penalty': penalty_type,
    'logreg__C': C_values
}

In [14]:
logreg_cv = GridSearchCV(
    logreg_pipeline, 
    param_grid,
    cv=kf,
    scoring=f1_macro,
    return_train_score=True
)

In [15]:
logreg_cv.fit(X_train, y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('logreg',
                                        LogisticRegression(solver='liblinear'))]),
             param_grid={'logreg__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03, 1.e+04,
       1.e+05]),
                         'logreg__penalty': ['l1', 'l2']},
             return_train_score=True,
             scoring=make_scorer(f1_score, average=macro))

In [16]:
best_poly_model = logreg_cv.best_estimator_
best_poly_score = logreg_cv.best_score_

print(f"Best Polynomial + LogisticRegression (ElasticNet) Macro F1: {best_poly_score:.4f}")
print("Best Params:", logreg_cv.best_params_)

Best Polynomial + LogisticRegression (ElasticNet) Macro F1: 0.2906
Best Params: {'logreg__C': 10.0, 'logreg__penalty': 'l2'}


In [17]:
spline_pipeline = Pipeline([
    ("spline", SplineTransformer(degree=3, include_bias=False)),
    ("logreg", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=10000))
])

param_grid_spline = {
    "spline__n_knots": [3, 4, 5, 6, 7]
}

grid_spline = GridSearchCV(spline_pipeline,
                           param_grid_spline, 
                           cv = kf, 
                           scoring = f1_macro)

In [18]:
grid_spline.fit(X_train, y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('spline',
                                        SplineTransformer(include_bias=False)),
                                       ('logreg',
                                        LogisticRegression(max_iter=10000))]),
             param_grid={'spline__n_knots': [3, 4, 5, 6, 7]},
             scoring=make_scorer(f1_score, average=macro))

In [19]:
best_spline_model = grid_spline.best_estimator_
best_spline_score = grid_spline.best_score_

print(f"Best Spline + LogisticRegression Macro F1: {best_spline_score:.4f}")
print("Best Params:", grid_spline.best_params_)

Best Spline + LogisticRegression Macro F1: 0.4530
Best Params: {'spline__n_knots': 7}


In [20]:
if best_poly_score > best_spline_score:
    final_model = best_poly_model
    print("Polynomial + Logistic Regression is Better.")
else:
    final_model = best_spline_model
    print("Spline + Logistic Regression is Better.")

Spline + Logistic Regression is Better.


In [21]:
y_test_pred = final_model.predict(X_test)

In [22]:
submission = pd.DataFrame({
    "id": ID,
    "cover_type": y_test_pred
})

In [23]:
submission

,id,cover_type
0,A200001,2
1,A200002,1
2,A200003,1
3,A200004,2
4,A200005,2
...,...,...
199995,A399996,1
199996,A399997,1
199997,A399998,2
199998,A399999,1


In [24]:
sample

,id,cover_type
0,A200001,4
1,A200002,6
2,A200003,6
3,A200004,3
4,A200005,1
...,...,...
199995,A399996,2
199996,A399997,4
199997,A399998,1
199998,A399999,5


In [25]:
submission.to_csv("submission.csv", index=False)